# Analisis de `suma_vectores.json`

In [ ]:
# Figuras VECTORIALES en vez de PNG. Tres motivos, en orden de importancia:
#   · git guarda cada blob comprimido, y un SVG de líneas comprime 5 veces
#     mejor que el mismo gráfico en PNG (150 KB contra 761 KB este notebook);
#   · se puede ampliar sin que se pixele;
#   · apagando la fecha que matplotlib incrusta en el SVG, dos ejecuciones
#     seguidas con los mismos datos dan bytes IDÉNTICOS — así reejecutar el
#     notebook no crea una copia nueva en la historia del repositorio.
%config InlineBackend.figure_formats = ['svg']
%config InlineBackend.print_figure_kwargs = {'metadata': {'Date': None}}

import json
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator

plt.rcParams["svg.hashsalt"] = "modulo05"   # ids estables entre corridas
plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.grid"] = True

# ── Ayudantes comunes a todas las secciones ─────────────────────────────────
#

print("Ayudantes cargados")

## 1. Análisis Temporal de las Señales

En esta sección cargamos los datos generados por la GPU desde `suma_vectores.json` y graficamos su evolución temporal muestra por muestra. 

Este gráfico nos permite observar visualmente la diferencia de comportamiento entre:
* La señal `x` (Ruido Uniforme entre -1.0 y 1.0).
* La señal `y` (Distribución Normal con media 2.0 y \(\sigma = 0.5\)).
* La señal resultante `z`, que hereda las propiedades de ambos procesos aleatorios.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cargar el JSON crudo para extraer metadatos escalares y vectores por separado
with open('./build/suma_vectores.json', 'r') as f:
    datos_crudos = json.load(f)

# Extraemos las variables de generación de forma genérica
x_min = datos_crudos['x_min']
x_max = datos_crudos['x_max']
y_mu = datos_crudos['y_mu']
y_sigma = datos_crudos['y_sigma']

# 2. Reconstruir el DataFrame de señales limpias utilizando las series correspondientes
df = pd.DataFrame({
    'x': datos_crudos['x'],
    'y': datos_crudos['y'],
    'z': datos_crudos['z']
}, index=datos_crudos['ix'])

# 3. Graficar las señales de manera automatizada usando las variables leídas
plt.figure(figsize=(15, 6))

plt.plot(df['x'], label=f'Señal X (Uniforme [{x_min}, {x_max}])', alpha=0.5, color='dodgerblue', linewidth=0.8)
plt.plot(df['y'], label=f'Señal Y (Normal $\\mu={y_mu}, \\sigma={y_sigma}$)', alpha=0.7, color='orange', linewidth=1.2)
plt.plot(df['z'], label='Resultado Z (X + Y)', alpha=0.8, color='crimson', linewidth=1.2)

plt.title('Comparativa Temporal de Señales Concurrentes en CUDA (Eje Dinámico)', fontsize=14)
plt.xlabel('Muestra ($i$)', fontsize=12)
plt.ylabel('Amplitud', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()


## 2. Análisis Estadístico y Frecuencial (Histogramas)

Cuando sumamos dos variables aleatorias independientes, la distribución del resultado ($Z$) matemáticamente corresponde a la **convolución** de las distribuciones de entrada. 

En este experimento:
* La distribución uniforme (cuadrada) actúa como un filtro suavizador suavizado sobre la campana de Gauss.
* El resultado final debería ser una **campana con hombros más planos y una base ensanchada** (una distribución normal deformada por los límites del ruido uniforme).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Obtener el mínimo y máximo absoluto de las 3 señales combinadas
min_absoluto = min(df['x'].min(), df['y'].min(), df['z'].min())
max_absoluto = max(df['x'].max(), df['y'].max(), df['z'].max())

# Agregamos un pequeño margen (padding) a los extremos para que no queden pegadas al borde
min_val = min_absoluto - 0.5
max_val = max_absoluto + 0.5

# Configurar las clases (bins) dinámicas compartidas para todo el rango visible
clases = np.linspace(min_val, max_val, 150) # Subido a 150 para cubrir bien el rango ensanchado

# 2. Graficar histogramas superpuestos
plt.figure(figsize=(12, 6.5))

plt.hist(df['x'], bins=clases, alpha=0.4, color='dodgerblue', edgecolor='blue', label=f'X: Uniforme [{x_min}, {x_max}]')
plt.hist(df['y'], bins=clases, alpha=0.4, color='orange', edgecolor='darkorange', label=f'Y: Normal ($\\mu={y_mu}, \\sigma={y_sigma}$)')
plt.hist(df['z'], bins=clases, alpha=0.5, color='crimson', edgecolor='darkred', histtype='bar', label='Z: Suma Vectorial (X + Y)')

# Anotación de la media real medida en Z
mu_z = df['z'].mean()
plt.axvline(mu_z, color='crimson', linestyle='--', linewidth=2, label=f'Media Medida Z = {mu_z:.2f}')

plt.title('Distribución Global de Frecuencias (Rango Dinámico Unificado)', fontsize=14)
plt.xlabel('Amplitud', fontsize=12)
plt.ylabel('Frecuencia (Conteos)', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()

# 3. Verificación de coincidencia matemática (Mantiene tus metadatos leídos del JSON)
media_teorica_z = ((x_min + x_max) / 2.0) + y_mu
varianza_teorica_x = ((x_max - x_min) ** 2) / 12.0
varianza_teorica_y = y_sigma ** 2
varianza_teorica_z = varianza_teorica_x + varianza_teorica_y

print(f"--- Validación Estadística Automática del Kernel ---")
print(f"Media esperada de Z (E[X] + E[Y]): {media_teorica_z:.4f} | Medida real: {mu_z:.4f}")
print(f"Varianza teórica de X: {varianza_teorica_x:.4f}         | Medida real: {df['x'].var():.4f}")
print(f"Varianza teórica de Y: {varianza_teorica_y:.4f}         | Medida real: {df['y'].var():.4f}")
print(f"Varianza esperada de Z (Var(X)+Var(Y)): {varianza_teorica_z:.4f} | Medida real: {df['z'].var():.4f}")
